# Time Series Course — m11-f2 (block 2)

_From the **Time Series & Forecasting** course on Zylo._

Run all cells (`Runtime → Run all` or `Ctrl+F9`) to execute.

In [ ]:
import torch
import torch.nn as nn
import math

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim=1, d_model=64, nhead=4,
                 num_layers=2, forecast_horizon=24, seq_len=168):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)

        # Positional encoding (sinusoidal)
        pe = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, forecast_horizon)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = self.input_projection(x) + self.pe[:, :x.size(1), :]
        x = self.encoder(x)           # (batch, seq_len, d_model)
        x = x.mean(dim=1)             # global average pool -> (batch, d_model)
        return self.head(x)            # (batch, forecast_horizon)

model = TimeSeriesTransformer()
dummy = torch.randn(32, 168, 1)  # batch=32, 168 hours, 1 feature
out = model(dummy)
print(f"Input:  {dummy.shape}")
print(f"Output: {out.shape}")  # (32, 24) — 24 hour forecast